# Google Analytics 4 to Databricks Pipeline Generation

This notebook generates Databricks Asset Bundle YAML files for GA4 ingestion pipelines using prefix+priority grouping logic.

## Process Overview
1. **Pipeline grouping**: Groups GA4 properties by prefix + priority combinations
2. **YAML generation**: Creates Databricks Asset Bundle YAML file with scheduled jobs

## Key Features
- **Prefix + Priority Grouping**: Each unique (prefix, priority) pair becomes a separate pipeline
- **Scheduled Jobs**: Automatically creates scheduled jobs based on cron expressions
- **Multi-table Support**: Each property can specify multiple tables (events, events_intraday, users)

## Prerequisites
- Upload the `load_balancing` and `deployment` directories to your Databricks workspace
- Ensure you have a CSV file with your GA4 properties
- Have a GA4 connection configured in Databricks

## Install Required Libraries

In [ ]:
%pip install pyyaml

In [ ]:
%load_ext autoreload
%autoreload 2

## Import Required Modules

In [ ]:
import pandas as pd
import os
from databricks.sdk import WorkspaceClient

# Import the complete pipeline generation function
from run_pipeline_generation import run_complete_pipeline_generation

## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!

In [ ]:
# Input CSV configuration
INPUT_CSV = 'load_balancing/examples/example_config.csv'  # Update with your CSV path

# Project configuration
PROJECT_NAME = 'ga4_ingestion'                # Project name for the bundle
DEFAULT_SCHEDULE = '0 */6 * * *'              # Cron schedule (every 6 hours)

# Initialize workspace client
w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/ga4/dab_deployment'
WORKSPACE_HOST = workspace_host

print(f"Configuration:")
print(f"  User: {username}")
print(f"  Workspace: {workspace_host}")
print(f"  Output directory: {OUTPUT_DIR}")

## Understanding the Input CSV

Your CSV should have the following columns:

**Required Columns:**
- `source_catalog`: GCP project ID
- `source_schema`: GA4 property (e.g., "analytics_123456789")
- `tables`: Comma-separated list of tables (e.g., "events,events_intraday,users")
- `target_catalog`: Target Databricks catalog
- `target_schema`: Target Databricks schema
- `prefix`: Business unit or logical grouping (e.g., "business_unit1")
- `priority`: Priority level (e.g., "01", "02")

**Optional Columns:**
- `schedule`: Cron schedule (uses default if not specified)

**Pipeline Grouping Logic:**
- Properties are grouped by unique combinations of (prefix, priority)
- Each group becomes a separate pipeline
- Example: All properties with prefix="business_unit1" and priority="01" go into one pipeline

## Run Pipeline Generation

In [ ]:
result_df = run_complete_pipeline_generation(
    input_csv=INPUT_CSV,
    project_name=PROJECT_NAME,
    workspace_host=WORKSPACE_HOST,
    output_dir=OUTPUT_DIR,
    default_schedule=DEFAULT_SCHEDULE
)

## Review Generated Configuration

In [ ]:
display(result_df)

## Summary Statistics

In [ ]:
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total GA4 properties: {len(result_df)}")
print(f"Total pipelines: {result_df['pipeline_group'].nunique()}")
print(f"\nProperties per pipeline group:")
print(result_df.groupby('pipeline_group').size())
print(f"\nBreakdown by prefix and priority:")
print(result_df.groupby(['prefix', 'priority']).size())
print("="*80)

## View Pipeline Groups

In [ ]:
for pipeline_group in sorted(result_df['pipeline_group'].unique()):
    group_df = result_df[result_df['pipeline_group'] == pipeline_group]
    prefix = group_df['prefix'].iloc[0]
    priority = group_df['priority'].iloc[0]
    print(f"\nPipeline Group {pipeline_group} ({prefix} - Priority {priority}):")
    print("  Properties:")
    for _, row in group_df.iterrows():
        tables = row['tables'].split(',') if pd.notna(row['tables']) else []
        print(f"    - {row['source_schema']}: {', '.join(tables)}")

## View Generated YAML

In [ ]:
print("="*80)
print("GENERATED YAML (resources/ga4_pipeline.yml)")
print("="*80)
yaml_path = f"{OUTPUT_DIR}/resources/ga4_pipeline.yml"
with open(yaml_path, 'r') as f:
    print(f.read())

## Next Steps

1. **Review Generated Files**: 
   - `{OUTPUT_DIR}/resources/ga4_pipeline.yml` - Pipeline and job definitions
   
2. **Verify Configuration**:
   - Check that catalog/schema match your target environment
   - Verify connection name (default: "ga4_connection")
   
3. **Ensure GA4 Connection Exists**:
   - Go to Databricks Workspace → Catalog → Connections
   - Create connection if needed: Type = Google Analytics
   - Connection name must match the value in YAML

4. **Create databricks.yml** (if not exists):
   ```yaml
   bundle:
     name: ga4_ingestion
   
   include:
     - resources/*.yml
   
   targets:
     dev:
       mode: development
       default: true
   ```

5. **Deploy Using Databricks Asset Bundles**:
   ```bash
   cd {OUTPUT_DIR}
   databricks bundle validate -t dev
   databricks bundle deploy -t dev
   ```

6. **Monitor**: Check your Databricks workspace for the deployed pipelines